### Importing necessary libraries

In [1]:
import time
import pandas as pd
from selenium import webdriver
from bs4 import BeautifulSoup
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By

### Importing and setting browser options
- creates a chrome browser in headless mode
- disable blink features is used to reduce detection of automation
- a driver is created from this options

In [6]:
options = Options()
options.add_argument("--headless=new")   # run headless
options.add_argument("--disable-blink-features=AutomationControlled")
driver = webdriver.Chrome(options=options)

### Visting airbnb url

In [7]:
url = "https://www.airbnb.co.in/s/Ahmadabad--Gujarat/homes?refinement_paths%5B%5D=%2Fhomes&acp_id=7e9afa3d-b634-47de-bd99-a37f62260f93&date_picker_type=calendar&checkin=2025-09-20&checkout=2025-09-21&adults=2&source=structured_search_input_header&search_type=autocomplete_click&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2025-09-01&monthly_length=3&monthly_end_date=2025-12-01&price_filter_input_type=2&price_filter_num_nights=1&channel=EXPLORE&place_id=ChIJIxcnN0CEXjkRobQIMyNYLpI&query=Ahmadabad%2C%20Gujarat&search_mode=regular_search&federated_search_session_id=c7816f42-b79d-413f-9f50-366e8beadd04&pagination_search=true&cursor=eyJzZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0IjoxOCwidmVyc2lvbiI6MX0%3D"
print(f"Loading: {url}")
driver.get(url)
time.sleep(5)

Loading: https://www.airbnb.co.in/s/Ahmadabad--Gujarat/homes?refinement_paths%5B%5D=%2Fhomes&acp_id=7e9afa3d-b634-47de-bd99-a37f62260f93&date_picker_type=calendar&checkin=2025-09-20&checkout=2025-09-21&adults=2&source=structured_search_input_header&search_type=autocomplete_click&flexible_trip_lengths%5B%5D=one_week&monthly_start_date=2025-09-01&monthly_length=3&monthly_end_date=2025-12-01&price_filter_input_type=2&price_filter_num_nights=1&channel=EXPLORE&place_id=ChIJIxcnN0CEXjkRobQIMyNYLpI&query=Ahmadabad%2C%20Gujarat&search_mode=regular_search&federated_search_session_id=c7816f42-b79d-413f-9f50-366e8beadd04&pagination_search=true&cursor=eyJzZWN0aW9uX29mZnNldCI6MCwiaXRlbXNfb2Zmc2V0IjoxOCwidmVyc2lvbiI6MX0%3D


### auto-scrolling options

In [8]:
last_h = driver.execute_script("return document.body.scrollHeight")
while True:
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(3)
    new_h = driver.execute_script("return document.body.scrollHeight")
    if new_h == last_h:
        break
    last_h = new_h


### convert page to beautiful soup

In [9]:
soup = BeautifulSoup(driver.page_source, "html.parser")
listings = soup.find_all("div", {"itemprop": "itemListElement"})


In [12]:
for listing in listings:
    title = listing.find("meta", {"itemprop": "name"})["content"]

In [13]:
subtitle_tag = listing.find("div", {"class": lambda x: x and "dir" in x})
subtitle = subtitle_tag.text.strip() if subtitle_tag else ""

In [14]:
link = listing.find("a")["href"]
if link.startswith("/"):
    link = "https://www.airbnb.co.in" + link

In [15]:
price = listing.find(string=lambda t: "₹" in t).strip()


In [16]:
rating_tag = listing.find("span", {"aria-label": True})
rating = rating_tag.text if rating_tag else ""


In [18]:
all_data =[]
all_data.append([title, subtitle, price, rating, link])


In [24]:
df = pd.DataFrame(all_data, columns=["Title", "Subtitle", "Price", "Rating", "URL"])
df.to_csv("ahmpg2.csv", index=False, encoding="utf-8-sig")


In [23]:
driver.quit()
print(f"✅ Scraping finished. {len(df)} rows saved.")

✅ Scraping finished. 1 rows saved.


In [25]:
df.head()

,Title,Subtitle,Price,Rating,URL
0,Cheerful 2 bedroom Villa n Garden South Bopal,SuperhostSuperhostVilla in AhmedabadCheerful 2...,"₹57,631","₹57,631",https://www.airbnb.co.in/rooms/604814068892508...
